# 07 — Dataset Transformation (Long → Wide Format)
**Spacecraft Telemetry Anomaly Detection | Stage 1**

---
**Goal:** Reshape telemetry from its raw long format (one row per reading) into a wide format  
(one row per timestamp, one column per parameter) — the structure required by ML models.

> Long format is efficient for storage and transmission.  
> Wide format is what machine learning models actually consume.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa',
                     'axes.titlesize':11, 'font.size':10})
os.makedirs('plots_v2', exist_ok=True)
os.makedirs('processed_v2', exist_ok=True)

telemetry = pd.read_csv('telemetry_train.csv')
telemetry['timestamp'] = pd.to_datetime(telemetry['timestamp'])
print('Loaded — shape:', telemetry.shape)

### 7.1 Long Format — What We Start With

In [ ]:
# Long format: each row is one sensor reading
# 10 000 rows = 50 parameters × 200 readings each
print('Long format shape:', telemetry[['timestamp','parameter','value']].shape)
print('Unique timestamps :', telemetry['timestamp'].nunique())
print('Unique parameters :', telemetry['parameter'].nunique())
print('\nFirst 12 rows (different parameters at the same timestamp appear in separate rows):')
display(telemetry[['timestamp','parameter','value']].head(12))

# NOTE: In long format, a single spacecraft state (all 50 sensors at one moment)
# is spread across 50 separate rows. No ML model can directly see the full state.

### 7.2 Pivot to Wide Format
Each row becomes one complete spacecraft state snapshot — all 50 parameters visible together.

In [ ]:
# pivot_table: index = timestamp rows, columns = parameter names, values = sensor readings
# aggfunc='mean': in case of duplicate (timestamp, parameter) pairs -- take the average
tel_wide = telemetry.pivot_table(
    index='timestamp',
    columns='parameter',
    values='value',
    aggfunc='mean'
).reset_index()

tel_wide.columns.name = None  # Remove MultiIndex label artifact
tel_wide = tel_wide.sort_values('timestamp').reset_index(drop=True)

print('Wide format shape:', tel_wide.shape)
print('  --> Rows = unique timestamps | Columns = timestamp + 50 parameters')
print('\nFirst 5 rows, first 8 parameter columns:')
display(tel_wide.iloc[:5, :8])

# RESULT: 10 000 unique timestamps → 10 000 rows
# Each row now represents the FULL spacecraft state at one moment in time
# This is the shape that Isolation Forest, OCSVM, and Autoencoders expect

### 7.3 Why NaN Appears — and How We Handle It

In [ ]:
# IMPORTANT: In round-robin polling, at any given timestamp, only ONE parameter
# is being measured. All other 49 parameters have no reading at that exact millisecond.
# Pivoting creates 49 NaN values for each row (49 parameters not measured at that instant).
param_cols = [c for c in tel_wide.columns if c != 'timestamp']

total_nan  = tel_wide[param_cols].isnull().sum().sum()
total_vals = tel_wide.shape[0] * len(param_cols)
print(f'NaN before fill  : {total_nan:,} out of {total_vals:,} cells')
print(f'Fill rate needed : {total_nan/total_vals*100:.1f}% of the matrix is missing')

# Forward-fill: use the most recent known reading for each parameter
# Then back-fill: handle any NaN remaining at the very start
tel_wide[param_cols] = tel_wide[param_cols].ffill().bfill()

print(f'NaN after fill   : {tel_wide[param_cols].isnull().sum().sum()}')
print('\n--> Forward-fill is physically justified: sensor value "holds" until next reading')
print('    This is equivalent to the ground station assuming the last known value persists')

### 7.4 NaN Pattern Visualisation — Before Fill
Red = missing. Shows the natural round-robin polling pattern.

In [ ]:
# Recreate without fill to show the raw NaN pattern
tel_sparse = telemetry.pivot_table(
    index='timestamp', columns='parameter', values='value', aggfunc='mean'
).reset_index()
tel_sparse.columns.name = None
sparse_cols = [c for c in tel_sparse.columns if c != 'timestamp']

null_mat = tel_sparse[sparse_cols].iloc[:60].isnull().astype(int)  # First 60 timestamps

fig, ax = plt.subplots(figsize=(20, 8))
sns.heatmap(null_mat.T,
            cmap=['#f8f9fa','#d62728'],  # white=present, red=missing
            cbar=False, ax=ax, linewidths=0.2, linecolor='#dee2e6')
ax.set_title('NaN Pattern — First 60 Timestamps Before Forward-Fill\n'
             'Red = missing (parameter not yet polled) | White = value present',
             fontweight='bold')
ax.set_xlabel('Timestamp index (chronological)')
ax.set_ylabel('Parameter')
plt.yticks(fontsize=6)
plt.tight_layout()
plt.savefig('plots_v2/07_nan_pattern.png', dpi=150, bbox_inches='tight')
plt.show()

# RESULT: Each parameter has only ONE white cell in the first 60 rows
# This is the round-robin sampling pattern -- the ground station polls each sensor in turn
# After forward-fill, each parameter's value propagates until the next actual reading

### 7.5 Long vs Wide — Format Comparison

In [ ]:
comparison = pd.DataFrame({
    'Property':    ['Rows','Columns','ML-ready','NaN handling required',
                    'One row represents','Suitable for'],
    'Long Format': [
        '10 000',
        '3  (timestamp, parameter, value)',
        'No — parameters in separate rows',
        'Not needed (no NaN in long format)',
        'One sensor reading',
        'Storage | transmission | long-format models (NCDE, Transformers)',
    ],
    'Wide Format': [
        f'{len(tel_wide):,}  (one per unique timestamp)',
        f'51  (timestamp + {len(param_cols)} parameters)',
        'Yes — all 50 sensors visible per row',
        'Forward-fill required (49 NaN per row before fill)',
        'Complete spacecraft state snapshot',
        'Isolation Forest | One-Class SVM | Autoencoder | PCA',
    ],
})
display(comparison)

In [ ]:
# Save the clean wide-format dataset
tel_wide.to_csv('processed_v2/telemetry_wide.csv', index=False)
print('Saved: processed_v2/telemetry_wide.csv')
print('Final shape:', tel_wide.shape, ' — ML-ready matrix')